# Suite EVL — Simplifiers

`Simplifiers.OfAny(term, scope)` simplifies a term with the variables in `scope` bound: what they decide folds into
literals, and the rest stays, the residual, a Patterns term. Its root is a literal when the scope decides the term:
`value` is then its value, and `evaluate()` gives it, or raises `NotEvaluated` naming why the root is not one. A term is
a value of `Expressions.Framework.Term`, built in a store, in a transaction (here through `term`, from a writer).
Simplification is three-valued (absent and incomparable values are unknown, and logic follows Kleene), never coerces
native types, and folds only the core operations: any other stays in the residual, named (`residues`).

In [ ]:
import math

from mbse.Expressions import Framework
from mbse.Expressions.IEEE1164 import Expressions as IEEE1164
from mbse.Expressions.IEEE754 import Expressions as IEEE754
from mbse.Expressions.ISOIEC11404 import Expressions as ISOIEC11404
from mbse.Expressions.Unicode import Expressions as Unicode
from mbse.Expressions.Patterns import Expressions as Patterns
from mbse.Expressions.Patterns.Expressions import Expressions as E, Simplifiers as V
from mbse.Expressions.Framework import Terms as F
from mbse.Schemas.Framework import Builders
from support import raises
from mbse.Expressions.Framework.Simplifiers import NotEvaluated

store = Builders.Store()
store.transaction(Framework.install)
store.transaction(ISOIEC11404.install)  # the standard dialects, before the dialects that use their kinds
store.transaction(IEEE754.install)
store.transaction(IEEE1164.install)
store.transaction(Unicode.install)
store.transaction(Patterns.install)
store.transaction(Patterns.define)  # Patterns' operations, which validation reads


def term(spec):
    """The term `spec` writes, the root of a holder built within the transaction open: validation reads its holder's
    store, and the term, a visitor of that transaction, lives within it."""
    return F.term(store.Expressions.Framework.Expression(lambda e: e.root(E.DIALECT.spec(spec))))


def ev(spec, **scope):
    """The value the simplification of the term `spec` writes folds to, with `scope` bound: None where its root is not a
    literal, unknown."""
    return V.OfAny(term(spec), scope).value


def same(simplified, expected):
    """Whether a simplification's residual is the term `expected` writes."""
    return F.same(term(simplified.residual), term(expected))


def people(txn):
    """A person, a reference object; a card that reaches its holder by phone or email, a union of value objects."""
    S = txn.Schemas
    return txn.uid("evl.people/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Person": S.OfObject, "Reach": S.OfUnion, "Code": S.OfUnion, "Card": S.OfObject})
        .spec([
            lambda _: _.Person.name("evl.Person").ref(True).properties({"name": S.OfStr, "age": S.OfInt}),
            lambda _: _.Reach.name("evl.Reach").branches([
                lambda t: t.name("phone").schema([S.OfObject(lambda o: o.properties({"kind": S.OfStr, "number": S.OfStr}))]),
                lambda t: t.name("email").schema([S.OfObject(lambda o: o.properties({"kind": S.OfStr, "address": S.OfStr}))]),
            ]),
            lambda _: _.Code.name("evl.Code").branches([  # a union of scalars
                lambda t: t.name("number").schema([S.OfInt]), lambda t: t.name("text").schema([S.OfStr]),
                lambda t: t.name("digits").schema([S.OfContainer(lambda c: c.index([S.OfInt]).element([S.OfInt]))])]),
            lambda _: _.Card.name("evl.Card").ref(True).properties({"reach": _.Reach, "code": _.Code}),
        ])]))])


store.transaction(people)


def persons():
    """`ann`, a person aged 30, and `nobody`, a person with no properties, the persons the cases below share, built anew
    within the transaction open, as an object read or built lives only within the transaction that made it."""
    return store.evl.Person(lambda p: p.name("Ann").age(30)), store.evl.Person(lambda p: p)

<a id="EVL-01"></a>
## EVL-01 · Simplifying literals, variables, lets, `get` and `has`

In [ ]:
with store.transaction():
    ann, nobody = persons()
    person = E.variable("person")
    assert ev(3) == 3 and ev(E.literal("x")) == "x"
    bound = V.OfAny(term(person), {"person": ann})  # an object has no literal: the variable stays, its value known
    assert not bound.literal and bound.value is None and same(bound, person) and bound.free == {"person"}
    assert ev(person.age, person=ann) == 30 and ev(person.age, person=nobody) is None  # absent: unknown
    assert ev(person.has("age"), person=ann) is True and ev(person.has("age"), person=nobody) is False
    assert ev(person.age, person=None) is None and ev(person.has("age"), person=None) is None  # unknown object
    assert ev(E.let_("a", person.age, E.variable("a").add(1)), person=ann) == 31
    assert ev(E.let_("a", 1, E.let_("a", 2, E.variable("a")))) == 2  # the innermost binding wins
    assert ev(E.let_("a", 1, E.variable("a").add(E.let_("a", 2, E.variable("a"))))) == 3  # and only in its body
    one, term_value = term(E.literal(1)), E.variable("term_value")
    assert ev(term_value.has("patterns"), term_value=one) is True  # terms are values too: a union value of their dialect
    assert ev(term_value.patterns.literal.value, term_value=one) == 1
    unbound = V.OfAny(term(E.variable("x")))  # not bound: it stays, and its root is no literal
    assert unbound.value is None and same(unbound, E.variable("x")) and unbound.residues == ()
    with raises(V.F.NotEvaluated, match="the root is not a closed constructor expression: x is not bound"):
        unbound.evaluate()
    assert V.OfAny(term(E.literal(1).add(2))).evaluate() == 3  # a literal root: evaluated
    with raises(TypeError, match="get expects an object, got int"):
        ev(E.operation("get", 1, "x"))
    with raises(TypeError, match="has expects a property name, got int"):
        ev(E.operation("has", person, 1), person=ann)

<a id="EVL-02"></a>
## EVL-02 · Comparisons are three-valued and never coerce

In [ ]:
with store.transaction():
    ann, nobody = persons()
    unit = E.literal(1)
    assert [ev(unit.lt(2)), ev(unit.le(1)), ev(unit.gt(2)), ev(unit.ge(1)), ev(unit.eq(1)), ev(unit.ne(1))] == [
        True, True, False, True, True, False]
    assert ev(E.literal("b").gt("a")) is True and ev(E.literal(b"\x01").lt(b"\x02")) is True
    assert ev(unit.eq(1.0)) is None and ev(unit.lt(1.5)) is None and ev(unit.ne("1")) is None  # different types
    assert ev(E.literal(True).eq(False)) is False and ev(E.literal(True).lt(False)) is None  # bools are not ordered
    assert ev(E.literal(True).le(True)) is True
    nan = E.literal(math.nan)
    assert ev(nan.eq(math.nan)) is True and ev(nan.eq(1.0)) is False and ev(nan.lt(1.0)) is None
    assert ev(E.literal(-0.0).lt(0.0)) is True and ev(E.literal(-0.0).eq(0.0)) is False
    age = E.variable("x").age
    assert ev(age.eq(1), x=nobody) is None and ev(age.ne(1), x=nobody) is None and ev(age.lt(1), x=nobody) is None
    first, second = E.variable("a"), E.variable("b")
    assert ev(first.eq(second), a=ann, b=ann) is True and ev(first.eq(second), a=ann, b=nobody) is False  # objects: by identity
    assert ev(first.lt(second), a=ann, b=nobody) is None and ev(first.eq(1), a=ann) is None

<a id="EVL-03"></a>
## EVL-03 · Logic follows Kleene; arithmetic keeps one type

In [ ]:
with store.transaction():
    T, F_, U = E.literal(True), E.literal(False), E.variable("u")  # u is unknown


    def table(method):
        return [[ev(getattr(x, method)(y), u=None) for y in (T, F_, U)] for x in (T, F_, U)]


    assert table("and_") == [[True, False, None], [False, False, False], [None, False, None]]
    assert table("or_") == [[True, True, True], [True, False, None], [True, None, None]]
    assert table("implies") == [[True, False, None], [True, True, True], [True, None, None]]
    assert [ev(x.not_(), u=None) for x in (T, F_, U)] == [False, True, None]
    missing = E.variable("missing")  # never evaluated when the first operand decides
    assert ev(F_.and_(missing)) is False and ev(T.or_(missing)) is True and ev(F_.implies(missing)) is True
    with raises(TypeError, match="and expects bool operands, got int"):
        ev(T.and_(1))
    with raises(TypeError, match="not expects bool operands, got str"):
        ev(E.literal("x").not_())
    pair = E.literal(2)
    assert [ev(pair.add(3)), ev(pair.sub(3)), ev(pair.mul(3)), ev(pair.neg())] == [5, -1, 6, -2]
    assert ev(E.literal(1.5).add(1.0)) == 2.5 and ev(E.literal(2**70).mul(2)) == 2**71
    assert ev(U.add(1), u=None) is None and ev(U.neg(), u=None) is None
    with raises(TypeError, match="add expects numbers of one domain, got int and float"):
        ev(E.literal(1).add(1.5))
    with raises(TypeError, match="sub expects numbers of one domain, got bool and bool"):
        ev(T.sub(T))
    with raises(TypeError, match="neg expects a number, got str"):
        ev(E.literal("x").neg())

<a id="EVL-04"></a>
## EVL-04 · Checking: `validate()` finds what simplification would leave open or raise

In [ ]:
with store.transaction():
    assert E.DIALECT.validate(term(E.operation("lt", 1))) == ["lt takes 2 arguments, got 1"]
    foo = term(E.operation("foo", 1))
    assert E.DIALECT.validate(foo) == E.DIALECT.validate(foo, core=True) == ["'foo' is not an operation Patterns defines"]
    person = E.variable("person")
    checked = term(E.let_("a", person.age, E.variable("a").ge(18).and_(person.has("email"))))
    assert E.DIALECT.validate(checked, bound={"person"}, core=True) == []
    assert E.DIALECT.validate(checked) == ["value: argument 0: variable 'person' is not bound",
                                           "body: argument 1: argument 0: variable 'person' is not bound"]
    kept = V.OfAny(foo)  # an operation Patterns does not define stays, named
    assert same(kept, E.operation("foo", 1)) and [r.reason for r in kept.residues] == ["Patterns call 'foo' is not simplified"]
    with raises(V.F.NotEvaluated, match="the root is not a closed constructor expression: Patterns call 'foo' is not simplified"):
        kept.evaluate()
    with raises(TypeError, match="lt takes 2 arguments, got 1"):
        ev(E.operation("lt", 1))
    with raises(ValueError, match="a literal needs a value"):
        ev(E.literal(None))
    with raises(ValueError, match="a let needs a value"):
        ev(E.let_("x", None, None))
    with raises(ValueError, match="a let needs a body"):
        ev(E.let_("x", 1, None))
    # A term is a nested value: written into another, it is copied, so a term never holds itself.
    inner = term(E.operation("not", True))
    outer = term(E.operation("not", inner))
    assert ev(outer) is True and len(list(F.walk(outer))) == 3 and len(list(F.walk(inner))) == 2
    with raises(TypeError, match="expected a term, a writer or a native value, got [7]"):
        E.operation("not", [7])

<a id="EVL-05"></a>
## EVL-05 · The union of terms names each branch by its dialect, and a dialect's `Term` each by its kind's tag

In [ ]:
with store.transaction():
    Term, Union = store.registered("Expressions.Framework.Term"), store.registered("Expressions.Patterns.Expressions.Term")
    assert [branch.name for branch in Term.branches] == ["patterns"] and Term.branches[0].schema is Union
    assert [branch.name for branch in Union.branches] == [
        "literal", "call", "variable", "let", "list", "mapping", "quantifier", "error", "missing"]
    assert [branch.schema.name for branch in Union.branches] == [
        "Expressions.Framework.Literal", "Expressions.Framework.Call", "Expressions.Framework.Variable",
        "Expressions.Framework.Let", "Expressions.Framework.List", "Expressions.Framework.Mapping",
        "Expressions.Patterns.Expressions.Quantifier", "Expressions.Framework.Error", "Expressions.Framework.Missing"]
    samples = [E.literal(1), E.operation("f", 1, "x"), E.variable("x"), E.let_("a", 1, 2), E.list_(1), E.mapping(("k", 1)),
               E.quantifier("all", "x", E.variable("xs"), True), lambda u: u.error(lambda r: r.text("?")),
               lambda u: u.missing(lambda r: r)]
    assert [F.kind_of(term(sample))[0] for sample in samples] == [branch.name for branch in Union.branches]
    assert Union.validate() == []
    invalid = [V.OfAny(term(samples[7])), V.OfAny(term(samples[8]))]  # invalid syntax stays, named: no value of its own
    assert [(r.dialect, r.kind, r.category) for s in invalid for r in s.residues] == [
        ("Patterns", "error", "b"), ("Patterns", "missing", "b")]
    assert [s.residues[0].reason for s in invalid] == ["Patterns error is not simplified", "Patterns missing is not simplified"]

<a id="EVL-06"></a>
## EVL-06 · One entry point simplifies a term of any kind, an expression or its root

In [ ]:
with store.transaction():
    x = E.variable("x")
    assert ev(3) == 3 and ev(E.literal("s")) == "s" and ev(x, x=2) == 2
    assert ev(x.add(1), x=1) == 2 and ev(E.let_("y", 1, E.variable("y").neg())) == -1
    assert ev(x.all("i", E.variable("i").gt(0)), x=[1, 2]) is True
    assert ev(x, x=None) is None
    expression = store.Expressions.Framework.Expression(lambda e: e.root(x.add(1)))
    assert V.OfAny(expression, {"x": 1}).value == 2 and V.OfAny(expression.root, {"x": 2}).value == 3  # an expression, its root
    with raises(TypeError, match="not a term: 3"):
        V.OfAny(3)
    # A value that a dialect writes as a closed constructor expression, not a literal, is an evaluation too
    from mbse.Expressions.Framework.Simplifiers import Simplifier
    built = Simplifier(V.RULES, literal=lambda value: None, construct=lambda value: E.literal(value))
    known = built.known(x, 3)
    assert known.constructed and not known.literal and known.value == 3
    simplified = built(term(x.add(1)), {"x": 1})
    assert simplified.constructed and not simplified.literal and simplified.evaluate() == 2

<a id="EVL-07"></a>
## EVL-07 · A constraint about a value, bound to a variable, reads value objects and union values

In [ ]:
with store.transaction():
    card, mail = E.variable("card"), E.variable("mail")  # a card, and an email, a value object
    good = store.evl.Card(lambda c: c.reach(lambda r: r.email(lambda e: e.kind("work").address("a@x"))))
    phone = store.evl.Card(lambda c: c.reach(lambda r: r.phone(lambda e: e.number("1"))))
    same = store.evl.Card(lambda c: c.reach(lambda r: r.email(lambda e: e.kind("work").address("a@x"))))
    home = store.evl.Card(lambda c: c.reach(lambda r: r.email(lambda e: e.kind("home").address("a@x"))))
    work_phone = store.evl.Card(lambda c: c.reach(lambda r: r.phone(lambda e: e.kind("work"))))
    email = card.get("reach").get("email")  # a union value is a record of its one branch
    assert ev(card.get("reach").has("email"), card=good) is True and ev(card.get("reach").has("phone"), card=good) is False
    assert ev(email.get("kind").eq("work"), card=good) is True
    assert ev(email.get("kind").eq("home"), card=phone) is None
    assert ev(mail.get("kind").eq("phone"), mail=good.reach.email) is False and ev(mail.has("kind"), mail=None) is None
    assert ev(mail.get("kind"), mail=good.reach.email) == "work"  # a value object's property
    assert ev(mail.eq(mail), mail=good.reach.email) is True  # a value object compares deeply, by its properties
    other = E.variable("other")
    reach = card.get("reach").eq(other.get("reach"))  # union values: the same branch, equal
    assert ev(reach, card=good, other=same) is True
    assert ev(reach, card=good, other=home) is False
    assert ev(reach, card=good, other=work_phone) is None  # another branch
    assert ev(card.eq(other), card=good, other=good.reach.email) is None and ev(card.eq(other), card=good, other=good) is True
    coded = store.evl.Card(lambda c: c.code(lambda r: r.number(7)))
    assert ev(card.get("code").get("number"), card=coded) == 7 and ev(card.get("code").has("text"), card=coded) is False
    listed = store.evl.Card(lambda c: c.code(lambda r: r.digits(lambda k: (k.at(0).as_scalar(1), k.at(1).as_scalar(2), k)[2])))
    assert ev(card.get("code").get("digits").count(), card=listed) == 2  # a union's container branch reads as a collection

<a id="EVL-08"></a>
## EVL-08 · A parameter reference is a leaf: a context (`Contexts.OfContext`) assigns its parameter a value that folds, apart from the variables, or a term read in its place; one neither assigned nor bound stays, named among the free parameters; one bound to an object is put as its reference where a literal's value stood, and stays, named, in a term's place; `Terms.same` compares two by name

In [ ]:
from mbse.Expressions.Framework import Symbolics as S
from mbse.Schemas.Framework import JSON, References


def context(**given):
    """A context (`Contexts.OfContext`) whose parameters are given by name: each assigned a value, or, given as
    `(reference,)`, bound to the object the reference designates."""
    def entry(name, target):
        if isinstance(target, tuple):
            return lambda p: p.name(name).object(target[0])
        return lambda p: p.name(name).value(lambda v: v.value(target))
    return store.Contexts.OfContext(lambda c: c.parameters({name: entry(name, t) for name, t in given.items()}))


def written(simplified, expected):
    """Whether a simplification's residual is the term `expected` writes (EVL-07 rebinds `same`)."""
    return F.same(term(simplified.residual), term(expected))


with store.transaction():
    x, n, m = E.variable("x"), *(store.References.OfParameter(lambda p, k=k: p.name(k)) for k in ("n", "m"))
    # A parameter its context assigns a value is replaced by the value's literal, and what it closes folds; a context
    # is taken where a scope is, its variables the scope's variables
    assert V.OfAny(term(x.add(n)), S.Variables({"x": 1}).within(context(n=2))).value == 3
    assert written(V.OfAny(term(x.add(n)), context(n=2)), x.add(2)) and V.OfAny(term(n), context(n=2)).value == 2
    both = store.Contexts.OfContext(lambda c: c.variables({"x": lambda v: v.name("x").value(lambda r: r.value(5))})
                                    .parameters({"n": lambda p: p.name("n").value(lambda r: r.value(2))}))
    assert V.OfAny(term(x.add(n)), both).value == 7
    # A parameter and a variable of one name never collide: the variable n is 10, the parameter n is 1
    assert V.OfAny(term(E.variable("n").add(n)), S.Variables({"n": 10}).within(context(n=1))).value == 11
    assert V.OfAny(term(E.let_("n", 5, E.variable("n").add(n))), context(n=1)).value == 6  # a let binds the variable
    # A parameter its context neither assigns nor binds stays, named among the free parameters, beside the free names
    stays = V.OfAny(term(x.add(n).add(m)), S.Variables({"x": 1}).within(context(m=4)))
    assert written(stays, E.literal(1).add(n).add(4)) and stays.parameters == {"n"} and stays.free == set()
    assert V.OfAny(term(n)).parameters == {"n"} and written(V.OfAny(term(n)), n)  # at the root, written back there
    with raises(NotEvaluated, match="the root is not a closed constructor expression: parameter 'n' is neither "
                                    "assigned nor bound"):
        stays.evaluate()
    # A parameter assigned a term is read in its place, where its place holds that term (any term, in Patterns' calls)
    plus = term(x.add(1))
    assert written(V.OfAny(term(E.operation("mul", n, 2)), context(n=plus)), E.operation("mul", x.add(1), 2))
    assert V.OfAny(term(E.operation("mul", n, 2)), S.Variables({"x": 3}).within(context(n=plus))).value == 8
    # A parameter bound to an object is substituted by the object's reference, put as it is where the parameter stood
    # in a literal's value, which holds any value: nothing resolved, nothing read
    schema = References.of(store, store.registered("evl.Person"))  # a schema object's reference: any object's would do
    held = lambda u: u.literal(lambda r: r.value(n))  # noqa: E731
    assert E.DIALECT.validate(term(held)) == [] and V.OfAny(term(held)).parameters == {"n"}
    put = V.OfAny(term(E.list_(held, x)), context(n=(schema,)))
    expected = E.list_(lambda u: u.literal(lambda r: r.value(schema)), x)
    json = lambda spec: JSON.ToJSON(store)(store.registered("Expressions.Framework.Term"), term(spec))  # noqa: E731
    assert json(put.residual) == json(expected) and '"References.OfObject"' in json(put.residual)
    assert put.parameters == set() and put.free == {"x"} and put.residues == ()
    assert V.OfAny(term(E.literal(1).add(held)), context(n=2)).value == 3  # a value assigned there folds
    # A term's place holds no object's reference: there the parameter stays, named by a residue of class (b)
    bound = V.OfAny(term(x.add(n)), S.Variables({"x": 1}).within(context(n=(schema,))))
    assert written(bound, E.literal(1).add(n)) and bound.parameters == set()
    assert [(r.reason, r.category) for r in bound.residues] == [
        ("Patterns parameter 'n' bound to an object, which its place does not hold is not simplified", "b")]
    # A scope's sub-context: the inner context's parameters replace the outer one's of their names
    inner = S.Variables().within(context(n=2, m=3)).within(context(n=4))
    assert (S.given(inner, "n"), S.given(inner, "m"), S.given(inner, "k")) == (("value", 4), ("value", 3), None)
    assert S.given(object(), "n") is None and S.Variables().within(context(n=(schema,))).parameters["n"][0] == "object"
    with raises(TypeError, match="a scope is within a context (Contexts.OfContext), not "):
        S.Variables().within({"n": 2})
    # Terms.same compares parameter references by name: two references to n are the same, n and m differ
    assert F.same(term(x.add(n)), term(x.add(store.References.OfParameter(lambda p: p.name("n")))))
    assert not F.same(term(x.add(n)), term(x.add(m))) and not F.same(term(x.add(n)), term(x.add(E.variable("n"))))
    # Read as a leaf of every dialect: a reference its scope resolves, which validation does not require bound
    leaf = F.term(term(x.add(n)))
    assert F.is_parameter(F.form(leaf).arguments[1]) and F.form(F.form(leaf).arguments[1]) == F.Form("parameter", {"name": "n"})
    assert E.DIALECT.validate(leaf, ["x"]) == [] and E.DIALECT.kind(F.form(leaf).arguments[1])[2] is F.PARAMETER_KIND
    assert F.dialect_of(term(n)) == "Expressions.Framework.Term" and E.DIALECT.kind(term(n))[0] == "parameter"
    with raises(TypeError, match="a parameter reference is written in a term's place, not as a term's value"):
        term(E.Writer(lambda u: F.spec(n)(u)))
    # A value with no literal, a union value, is known: what it closes folds, and the parameter itself stays, needing
    # nothing
    code = store.evl.Card(lambda c: c.code(lambda r: r.number(4))).code
    coded = V.OfAny(term(n), context(n=code))
    assert written(coded, n) and coded.parameters == set() and coded.residues == ()
    assert V.OfAny(term(E.operation("get", n, "number")), context(n=code)).value == 4
    # A parameter its context names but neither assigns nor binds is free; a parameter reference has no node
    named = store.Contexts.OfContext(lambda c: c.parameters({"n": lambda p: p.name("n")}))
    assert S.given(S.Variables().within(named), "n") is None and V.OfAny(term(n), named).parameters == {"n"}
    with raises(S.Unbound, match="n"):
        F.node(*F.kind_of(F.form(leaf).arguments[1]))